# Strategy 2 - Optimal K Selection and Fair Clustering Evaluation

This notebook implements an approach for determining optimal k values and evaluates three clustering methods:
- Standard K-Means
- Fairlet Decomposition
- Binary Fair K-Means (BFKM)

Datasets: Adult, COMPAS, German Credit, Default Credit Card, Law School

In [2]:
import numpy as np
import pandas as pd
import pickle
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    adjusted_mutual_info_score,
    fowlkes_mallows_score,
    v_measure_score
)
from scipy.spatial.distance import cdist
from scipy.stats import entropy
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
import warnings

warnings.filterwarnings('ignore')
# Set random seed for reproducibility
np.random.seed(0)

## 1. Data Loading and Preprocessing

In [4]:
# DATA LOADING
def load_adult():
    print("\n" + "="*80)
    print("Loading Adult Income Dataset")
    print("="*80)
    
    columns = [
        'age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
        'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
        'hours-per-week', 'native-country', 'income'
    ]
    
    adult = pd.read_csv("./Dataset/adult/adult.data", names=columns, header=None)
    print(f"Shape: {adult.shape}")
    
    #remove rows with missing values
    adult = adult.replace(' ?', np.nan)
    adult = adult.dropna()
    
    # Sensitive attribute - gender
    adult['sex_binary'] = (adult['sex'].str.strip() == 'Male').astype(int)
    
    # Features for clustering (exclude sensitive and target)
    adult_features = [c for c in adult.columns if c not in ["income", "sex", "race", "sex_binary"]]
    
    print(f"Sex distribution: {adult['sex'].value_counts().to_dict()}")
    print(f"Binary encoding - 0 (Female): {(adult['sex_binary']==0).sum()}, 1 (Male): {(adult['sex_binary']==1).sum()}")
    
    return adult, adult_features, 'sex_binary'


def load_compas():
    print("\n" + "="*80)
    print("Loading COMPAS Dataset")
    print("="*80)
    
    compas = pd.read_csv("./Dataset/compas/compas-scores.csv")
    print(f"Shape: {compas.shape}")
    
    # Filter data
    compas = compas[
        (compas['days_b_screening_arrest'] <= 30) &
        (compas['days_b_screening_arrest'] >= -30) &
        (compas['is_recid'] != -1)
    ]
    
    # Remove missing values columns
    selected_cols = compas.columns[compas.isnull().sum() == 0].tolist()
    compas = compas[selected_cols]
    
    print(f"Shape after filtering: {compas.shape}")
    
    # Sensitive attribute - gender
    compas['sex_binary'] = (compas['sex'] == 'Male').astype(int)
    
    compas_features = [c for c in compas.columns if c not in ["is_recid", "sex", "race", "sex_binary"]]
    print(f"Race distribution:\n{compas['sex'].value_counts()}")
    print(f"Binary encoding - 0 (Female): {(compas['sex_binary']==0).sum()}, 1 (Male): {(compas['sex_binary']==1).sum()}")
    
    return compas, compas_features, 'sex_binary'


def load_german():
    print("\n" + "="*80)
    print("Loading German Credit Dataset")
    print("="*80)
    
    german_columns = [
        'checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount',
        'savings_status', 'employment', 'installment_rate', 'personal_status_sex',
        'other_parties', 'residence_since', 'property_magnitude', 'age',
        'other_payment_plans', 'housing', 'existing_credits', 'job',
        'num_dependents', 'own_telephone', 'foreign_worker', 'class'
    ]
    
    german = pd.read_csv("./Dataset/german/german.data", 
                         sep=' ', 
                         names=german_columns, 
                         header=None)
    
    print(f"Shape: {german.shape}")
    
    # Extract gender from personal_status_sex
    def extract_sex(status):
        if status in ['A91', 'A93', 'A94']:
            return 'male'
        elif status in ['A92', 'A95']:
            return 'female'
        return 'unknown'
    
    german['sex'] = german['personal_status_sex'].apply(extract_sex)
    german = german[german['sex'] != 'unknown']  # Remove unknown
    german['sex_binary'] = (german['sex'] == 'male').astype(int)
    
    german_features = [c for c in german.columns if c not in ["class", "sex", "sex_binary"]]
    
    print(f"Sex distribution: {german['sex'].value_counts().to_dict()}")
    print(f"Binary encoding - 0 (Female): {(german['sex_binary']==0).sum()}, 1 (Male): {(german['sex_binary']==1).sum()}")
    
    return german, german_features, 'sex_binary'


def load_credit():
    print("\n" + "="*80)
    print("Loading Default Credit Card Dataset")
    print("="*80)
    
    credit = pd.read_csv("./Dataset/default of credit card clients/default of credit card clients.csv",
                         header=0)
    
    # Rename columns
    if 'X1' in credit.columns or credit.columns[0] == 'ID':
        credit.columns = [
            'ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
            'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',
            'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
            'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
            'default'
        ]
    
    credit = credit.drop(columns=['ID'])
    print(f"Shape: {credit.shape}")
    
    # Extract gender
    credit['SEX'] = pd.to_numeric(credit['SEX'], errors='coerce')
    credit['sex'] = credit['SEX'].map({1.0: 'male', 2.0: 'female'})
    credit = credit.dropna(subset=['sex'])
    
    credit['sex_binary'] = (credit['sex'] == 'male').astype(int)
    credit_features = [c for c in credit.columns if c not in ["default", "sex", "SEX", "sex_binary"]]
    
    print(f"Sex distribution: {credit['sex'].value_counts().to_dict()}")
    print(f"Binary encoding - 0 (Female): {(credit['sex_binary']==0).sum()}, 1 (Male): {(credit['sex_binary']==1).sum()}")
    
    return credit, credit_features, 'sex_binary'


def load_law():
    print("\n" + "="*80)
    print("Loading Law School (LSAC) Dataset")
    print("="*80)
    
    lsac = pd.read_csv("./Dataset/law/law_dataset.csv")
    print(f"LSAC shape: {lsac.shape}")
    print(f"Columns: {lsac.columns.tolist()}")
    
    # Check male column
    print(f"\nMale column dtype: {lsac['male'].dtype}")
    print(f"Male values: {lsac['male'].unique()}")
    print(f"Male distribution:\n{lsac['male'].value_counts()}")
    
    # Convert male column to standard gender column
    lsac['sex'] = lsac['male'].map({
        1: 'Male',
        0: 'Female',
        1.0: 'Male',
        0.0: 'Female'
    })
    
    print(f"\nStandardized sex distribution:")
    print(lsac['sex'].value_counts())
    
    print(f"\nMissing values:")
    missing = lsac.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0].sort_values(ascending=False))
    else:
        print("✓ No missing values!")
    
    if lsac['sex'].isnull().any():
        print(f"\n Warning: {lsac['sex'].isnull().sum()} rows have null sex values")
        print("These will be dropped")
        lsac = lsac[lsac['sex'].notna()]
    
    lsac_clean = lsac.copy()
    print(f"\nCleaned data shape: {lsac_clean.shape}")
    

    lsac_clean['sex_binary'] = (lsac_clean['sex'] == 'Male').astype(int)
    
    # Define features
    drop_cols = ['male', 'sex', 'pass_bar', 'sex_binary']
    lsac_features = [c for c in lsac_clean.columns if c not in drop_cols]
    
    print(f"Binary encoding - 0 (Female): {(lsac_clean['sex_binary']==0).sum()}, 1 (Male): {(lsac_clean['sex_binary']==1).sum()}")
    return lsac_clean, lsac_features, 'sex_binary'

In [5]:
def preprocess_dataset(df, feature_cols, sensitive_col):
    # Extract features
    X_df = df[feature_cols].copy()
    
    # Encode categorical variables
    categorical_cols = X_df.select_dtypes(include=['object']).columns.tolist()
    
    if categorical_cols:
        print(f"  Encoding {len(categorical_cols)} categorical columns...")
        le = LabelEncoder()
        for col in categorical_cols:
            X_df[col] = le.fit_transform(X_df[col].astype(str))
    
    # Convert to numpy array
    X = X_df.values.astype(float)
    
    # Check for any remaining non-numeric values
    if np.any(np.isnan(X)) or np.any(np.isinf(X)):
        print("  Warning: Found NaN or Inf values, filling with 0")
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Standardize features
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    
    # Extract sensitive attribute
    sensitive_attr = df[sensitive_col].values.astype(int)
    
    print(f"  Final shape: {X.shape}")
    print(f"  Sensitive attribute distribution: 0={np.sum(sensitive_attr==0)}, 1={np.sum(sensitive_attr==1)}")
    
    return X, sensitive_attr

In [6]:
# Load all datasets
print("\n" + "="*80)
print("LOADING ALL DATASETS")
print("="*80)

datasets = {}
dataset_loaders = [
    ('adult', load_adult),
    ('compas', load_compas),
    ('german', load_german),
    ('credit', load_credit),
    ('law', load_law)
]

for name, loader in dataset_loaders:
    try:
        df, features, sensitive_col = loader()
        print(f"\nProcessing {name.upper()}...")
        X, sensitive = preprocess_dataset(df, features, sensitive_col)
        datasets[name] = {
            'X': X,
            'sensitive': sensitive,
            'n_samples': X.shape[0],
            'n_features': X.shape[1]
        }
    except Exception as e:
        print(f"  Error loading {name}: {e}")

print("\n" + "="*80)
print("DATASET SUMMARY")
print("="*80)
for name, data in datasets.items():
    print(f"{name.upper()}: {data['n_samples']} samples, {data['n_features']} features")

dataset_names = list(datasets.keys())
print(f"\nTotal datasets loaded: {len(dataset_names)}")


LOADING ALL DATASETS

Loading Adult Income Dataset
Shape: (32561, 15)
Sex distribution: {' Male': 20380, ' Female': 9782}
Binary encoding - 0 (Female): 9782, 1 (Male): 20380

Processing ADULT...
  Encoding 6 categorical columns...
  Final shape: (30162, 12)
  Sensitive attribute distribution: 0=9782, 1=20380

Loading COMPAS Dataset
Shape: (11757, 47)
Shape after filtering: (9395, 30)
Race distribution:
sex
Male      7462
Female    1933
Name: count, dtype: int64
Binary encoding - 0 (Female): 1933, 1 (Male): 7462

Processing COMPAS...
  Encoding 15 categorical columns...
  Final shape: (9395, 27)
  Sensitive attribute distribution: 0=1933, 1=7462

Loading German Credit Dataset
Shape: (1000, 21)
Sex distribution: {'male': 690, 'female': 310}
Binary encoding - 0 (Female): 310, 1 (Male): 690

Processing GERMAN...
  Encoding 13 categorical columns...
  Final shape: (1000, 20)
  Sensitive attribute distribution: 0=310, 1=690

Loading Default Credit Card Dataset
Shape: (30001, 24)
Sex distrib

## 2. CLUSTERING ALGORITHMS

In [8]:
# 1. STANDARD K-MEANS
def standard_kmeans(X, k, random_state=42):
    kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=10)
    labels = kmeans.fit_predict(X)
    centers = kmeans.cluster_centers_
    return labels, centers


# 2. FAIRLET DECOMPOSITION
def create_fairlets(X, sensitive_attr, p, q):
    n = len(X)
    group0_idx = np.where(sensitive_attr == 0)[0]
    group1_idx = np.where(sensitive_attr == 1)[0]
    
    fairlets = []
    fairlet_labels = np.full(n, -1)
    fairlet_id = 0
    
    used_0 = set()
    used_1 = set()
    
    # Create balanced fairlets
    while len(used_0) + p <= len(group0_idx) and len(used_1) + q <= len(group1_idx):
        # Select p points from group 0
        available_0 = [i for i in group0_idx if i not in used_0]
        selected_0 = np.random.choice(available_0, p, replace=False)
        
        # Select q points from group 1
        available_1 = [i for i in group1_idx if i not in used_1]
        selected_1 = np.random.choice(available_1, q, replace=False)
        
        # Create fairlet
        fairlet_members = np.concatenate([selected_0, selected_1])
        fairlets.append(fairlet_members)
        fairlet_labels[fairlet_members] = fairlet_id
        
        used_0.update(selected_0)
        used_1.update(selected_1)
        fairlet_id += 1
    
    # Handle remaining points
    remaining = np.where(fairlet_labels == -1)[0]
    if len(remaining) > 0:
        fairlets.append(remaining)
        fairlet_labels[remaining] = fairlet_id
    
    return fairlets, fairlet_labels


def fairlet_clustering(X, sensitive_attr, k, p=1, q=1):
    fairlets, fairlet_labels = create_fairlets(X, sensitive_attr, p, q)
    
    fairlet_centers = []
    for i, fairlet in enumerate(fairlets):
        center = np.mean(X[fairlet], axis=0)
        fairlet_centers.append(center)
    fairlet_centers = np.array(fairlet_centers)
    
    # Cluster fairlet centers
    if len(fairlet_centers) >= k:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        fairlet_cluster_labels = kmeans.fit_predict(fairlet_centers)
    else:
        fairlet_cluster_labels = np.arange(len(fairlet_centers))
    
    # Map points to clusters
    labels = np.zeros(len(X), dtype=int)
    for i, cluster_label in enumerate(fairlet_cluster_labels):
        labels[fairlets[i]] = cluster_label
    
    # Compute final cluster centers
    centers = []
    for i in range(k):
        cluster_points = X[labels == i]
        if len(cluster_points) > 0:
            centers.append(np.mean(cluster_points, axis=0))
        else:
            centers.append(np.zeros(X.shape[1]))
            
    return labels, np.array(centers)


# 3. BINARY FAIR K-MEANS (BFKM)
def bfkm_clustering(X, sensitive_attr, k, max_iter=100, balance_weight=1.0):
    n_samples = len(X)
    
    # Initialize centers randomly
    center_indices = np.random.choice(n_samples, k, replace=False)
    centers = X[center_indices].copy()
    
    for iteration in range(max_iter):
        old_centers = centers.copy()
        labels = np.zeros(n_samples, dtype=int)
        distances = cdist(X, centers, metric='euclidean')
        
        for i in range(k):
            labels[np.argmin(distances, axis=1) == i] = i
        
        for cluster_id in range(k):
            cluster_mask = (labels == cluster_id)
            cluster_sensitive = sensitive_attr[cluster_mask]
            
            if len(cluster_sensitive) == 0:
                continue
            
            n_group0 = np.sum(cluster_sensitive == 0)
            n_group1 = np.sum(cluster_sensitive == 1)
            
            # Calculate imbalance
            total_group0 = np.sum(sensitive_attr == 0)
            total_group1 = np.sum(sensitive_attr == 1)
            
            if total_group0 > 0 and total_group1 > 0:
                expected_ratio = total_group1 / total_group0
                actual_ratio = n_group1 / max(n_group0, 1)
                
                if abs(actual_ratio - expected_ratio) > 0.3:
                    cluster_indices = np.where(cluster_mask)[0]
                    
                    if actual_ratio > expected_ratio:
                        group1_in_cluster = cluster_indices[sensitive_attr[cluster_indices] == 1]
                        if len(group1_in_cluster) > 0:
                            dists = distances[group1_in_cluster, cluster_id]
                            farthest_idx = group1_in_cluster[np.argmax(dists)]
                            sorted_clusters = np.argsort(distances[farthest_idx])
                            labels[farthest_idx] = sorted_clusters[1] if sorted_clusters[1] != cluster_id else sorted_clusters[2]
                    else:
                        group0_in_cluster = cluster_indices[sensitive_attr[cluster_indices] == 0]
                        if len(group0_in_cluster) > 0:
                            dists = distances[group0_in_cluster, cluster_id]
                            farthest_idx = group0_in_cluster[np.argmax(dists)]
                            sorted_clusters = np.argsort(distances[farthest_idx])
                            labels[farthest_idx] = sorted_clusters[1] if sorted_clusters[1] != cluster_id else sorted_clusters[2]
        
        # Update centers
        for i in range(k):
            cluster_points = X[labels == i]
            if len(cluster_points) > 0:
                centers[i] = np.mean(cluster_points, axis=0)
        
        # Check convergence
        if np.allclose(centers, old_centers, rtol=1e-4):
            break
    
    return labels, centers

## 3. QUALITY AND FAIRNESS METRICS

In [10]:
def calculate_quality_metrics(X, labels):
    k = len(np.unique(labels))
    n_samples = len(X)

    
    # Inertia (SSE / WCSS equivalent)
    cluster_centers = []
    inertia = 0
    
    for cluster_id in range(k):
        cluster_mask = (labels == cluster_id)
        cluster_points = X[cluster_mask]
        
        if len(cluster_points) > 0:
            center = np.mean(cluster_points, axis=0)
            cluster_centers.append(center)
            inertia += np.sum((cluster_points - center) ** 2)
        else:
            cluster_centers.append(np.zeros(X.shape[1]))
    
    cluster_centers = np.array(cluster_centers)

    
    # Silhouette Score
    if k > 1 and k < len(X):
        try:
            sil_score = silhouette_score(X, labels)
        except:
            sil_score = 0
    else:
        sil_score = 0
    

    # Calinski-Harabasz Index
    if k > 1 and k < len(X):
        try:
            ch_score = calinski_harabasz_score(X, labels)
        except:
            ch_score = 0
    else:
        ch_score = 0

    
    # Davies-Bouldin Index
    if k > 1 and k < len(X):
        try:
            db_score = davies_bouldin_score(X, labels)
        except:
            db_score = 0
    else:
        db_score = 0
    

    # Dunn Index
    dunn_index = 0
    if k > 1:
        try:
            # Minimum inter-cluster distance
            inter_cluster_dists = cdist(cluster_centers, cluster_centers, metric='euclidean')
            np.fill_diagonal(inter_cluster_dists, np.inf)
            min_inter_dist = np.min(inter_cluster_dists)
            
            # Maximum intra-cluster distance (diameter)
            max_intra_dist = 0.0
            for cluster_id in range(k):
                cluster_points = X[labels == cluster_id]
                if len(cluster_points) > 1:
                    pairwise_dists = pdist(cluster_points, metric='euclidean')
                    if len(pairwise_dists) > 0:
                        max_intra_dist = max(max_intra_dist, np.max(pairwise_dists))
            
            if max_intra_dist > 0:
                dunn_index = min_inter_dist / max_intra_dist
        except:
            dunn_index = 0
    

    # WCSS, BCSS, TSS, Variance Ratio
    overall_mean = np.mean(X, axis=0)
    tss = np.sum((X - overall_mean) ** 2)
    
    wcss = 0.0
    bcss = 0.0
    
    for cluster_id in range(k):
        cluster_points = X[labels == cluster_id]
        n_c = len(cluster_points)
        
        if n_c > 0:
            cluster_mean = np.mean(cluster_points, axis=0)
            
            # WCSS
            wcss += np.sum((cluster_points - cluster_mean) ** 2)
            
            # BCSS
            bcss += n_c * np.sum((cluster_mean - overall_mean) ** 2)
    
    variance_ratio = bcss / wcss if wcss > 0 else 0.0
    
  
    # Inter-Cluster Distance
    inter_cluster_distance = 0
    if k > 1:
        try:
            pairwise_dists = pdist(cluster_centers, metric='euclidean')
            inter_cluster_distance = np.mean(pairwise_dists) if len(pairwise_dists) > 0 else 0
        except:
            inter_cluster_distance = 0
    

    # Intra-Cluster Distance
    intra_distances = []
    for cluster_id in range(k):
        cluster_points = X[labels == cluster_id]
        if len(cluster_points) > 1:
            try:
                pairwise_dists = pdist(cluster_points, metric='euclidean')
                intra_distances.append(np.mean(pairwise_dists))
            except:
                pass
    
    intra_cluster_distance = np.mean(intra_distances) if intra_distances else 0
    

    # Separation Index
    separation_index = 0
    if intra_cluster_distance > 0:
        separation_index = inter_cluster_distance / intra_cluster_distance
    

    # Return all
    return {
        'Inertia': inertia,
        'Silhouette': sil_score,
        'Calinski_Harabasz': ch_score,
        'Davies_Bouldin': db_score,
        'Dunn_Index': dunn_index,
        'WCSS': wcss,
        'BCSS': bcss,
        'TSS': tss,
        'Variance_Ratio': variance_ratio,
        'Inter_Cluster_Distance': inter_cluster_distance,
        'Intra_Cluster_Distance': intra_cluster_distance,
        'Separation_Index': separation_index
    }



# EXTERNAL METRICS
def calculate_external_metrics(labels_true, labels_pred):
    try:
        ari = adjusted_rand_score(labels_true, labels_pred)
    except:
        ari = 0.0
    
    try:
        nmi = normalized_mutual_info_score(labels_true, labels_pred)
    except:
        nmi = 0.0
    
    try:
        ami = adjusted_mutual_info_score(labels_true, labels_pred)
    except:
        ami = 0.0
    
    try:
        fmi = fowlkes_mallows_score(labels_true, labels_pred)
    except:
        fmi = 0.0
    
    try:
        v_measure = v_measure_score(labels_true, labels_pred)
    except:
        v_measure = 0.0
    
    return {
        'ARI': ari,
        'NMI': nmi,
        'AMI': ami,
        'FMI': fmi,
        'V-Measure': v_measure
    }


# FAIRNESS METRICS
# Balance
def calculate_balance(labels, sensitive_attr):
    k = len(np.unique(labels))
    balances = []
    
    for cluster_id in range(k):
        cluster_mask = (labels == cluster_id)
        cluster_sensitive = sensitive_attr[cluster_mask]
        
        if len(cluster_sensitive) == 0:
            continue
        
        n_group0 = np.sum(cluster_sensitive == 0)
        n_group1 = np.sum(cluster_sensitive == 1)
        
        if n_group0 > 0 and n_group1 > 0:
            balance = min(n_group0, n_group1) / max(n_group0, n_group1)
            balances.append(balance)
    
    return np.mean(balances) if balances else 0.0


# Statistical Parity Difference
def calculate_spd(labels, sensitive_attr):
    k = len(np.unique(labels))
    total_group0 = np.sum(sensitive_attr == 0)
    total_group1 = np.sum(sensitive_attr == 1)
    
    spd_values = []
    
    for cluster_id in range(k):
        cluster_mask = (labels == cluster_id)
        n_group0 = np.sum((sensitive_attr == 0) & cluster_mask)
        n_group1 = np.sum((sensitive_attr == 1) & cluster_mask)
        
        p0 = n_group0 / total_group0 if total_group0 > 0 else 0
        p1 = n_group1 / total_group1 if total_group1 > 0 else 0
        
        spd = abs(p0 - p1)
        spd_values.append(spd)
    
    return np.mean(spd_values)


# Disparate Impact
def calculate_disparate_impact(labels, sensitive_attr):
    k = len(np.unique(labels))
    total_group0 = np.sum(sensitive_attr == 0)
    total_group1 = np.sum(sensitive_attr == 1)
    
    di_values = []
    
    for cluster_id in range(k):
        cluster_mask = (labels == cluster_id)
        n_group0 = np.sum((sensitive_attr == 0) & cluster_mask)
        n_group1 = np.sum((sensitive_attr == 1) & cluster_mask)
        
        p0 = n_group0 / total_group0 if total_group0 > 0 else 0
        p1 = n_group1 / total_group1 if total_group1 > 0 else 0
        
        if p0 > 0 and p1 > 0:
            di = min(p0, p1) / max(p0, p1)
            di_values.append(di)
    
    return np.mean(di_values) if di_values else 0.0


# Entropy
def calculate_entropy(labels, sensitive_attr):
    k = len(np.unique(labels))
    entropies = []
    
    for cluster_id in range(k):
        cluster_mask = (labels == cluster_id)
        cluster_sensitive = sensitive_attr[cluster_mask]
        
        if len(cluster_sensitive) == 0:
            continue
        
        n_group0 = np.sum(cluster_sensitive == 0)
        n_group1 = np.sum(cluster_sensitive == 1)
        
        if n_group0 > 0 and n_group1 > 0:
            probs = np.array([n_group0, n_group1]) / len(cluster_sensitive)
            ent = entropy(probs, base=2)
            entropies.append(ent)
    
    # Normalize by max entropy
    return np.mean(entropies) if entropies else 0.0


# EVALUATION FUNCTION
def evaluate_clustering(X, labels, sensitive_attr=None, labels_true=None):
    results = {}
    
    # Quality metrics (Internal - 12个)
    results['quality'] = calculate_quality_metrics(X, labels)
    
    # External metrics (5个)
    if labels_true is not None:
        results['external'] = calculate_external_metrics(labels_true, labels)
    
    # Fairness metrics (4个)
    if sensitive_attr is not None:
        results['fairness'] = {
            'Balance': calculate_balance(labels, sensitive_attr),
            'SPD': calculate_spd(labels, sensitive_attr),
            'Disparate_Impact': calculate_disparate_impact(labels, sensitive_attr),
            'Entropy': calculate_entropy(labels, sensitive_attr)
        }
    
    return results


# EXPERIMENT FRAMEWORK
def run_clustering_experiment(dataset_name, X, sensitive_attr, k, method_name, 
                              labels_true=None, method_function=None):
    # Run clustering
    if method_function is None:
        raise ValueError("method_function must be provided")
    
    labels, centers = method_function(X, sensitive_attr, k)
    
    # Calculate all metrics
    metrics = evaluate_clustering(X, labels, sensitive_attr, labels_true)
    
    return {
        'dataset': dataset_name,
        'method': method_name,
        'k': k,
        'labels': labels,
        'centers': centers,
        'quality': metrics['quality'],
        'external': metrics.get('external', {}),
        'fairness': metrics.get('fairness', {})
    }


def create_comparison_tables(results, approach_name):
    dataset_names = list(results.keys())
    methods = list(next(iter(results.values())).keys())

    # Quality Metrics Table
    quality_data = []
    for dataset_name in dataset_names:
        for method in methods:
            if method in results[dataset_name]:
                res = results[dataset_name][method]
                quality_data.append({
                    'Dataset': dataset_name.upper(),
                    'Method': method.upper(),
                    'k': res['k'],
                    'Inertia': res['quality']['Inertia'],
                    'Silhouette': res['quality']['Silhouette'],
                    'Calinski_Harabasz': res['quality']['Calinski_Harabasz'],
                    'Davies_Bouldin': res['quality']['Davies_Bouldin'],
                    'Dunn_Index': res['quality']['Dunn_Index'],
                    'WCSS': res['quality']['WCSS'],
                    'BCSS': res['quality']['BCSS'],
                    'TSS': res['quality']['TSS'],
                    'Variance_Ratio': res['quality']['Variance_Ratio'],
                    'Inter_Cluster_Distance': res['quality']['Inter_Cluster_Distance'],
                    'Intra_Cluster_Distance': res['quality']['Intra_Cluster_Distance'],
                    'Separation_Index': res['quality']['Separation_Index']
                })
    
    quality_df = pd.DataFrame(quality_data)
    
    # External Metrics Table
    for dataset_name in dataset_names:
        for method in methods:
            if method in results[dataset_name] and results[dataset_name][method]['external']:
                res = results[dataset_name][method]
                external_data.append({
                    'Dataset': dataset_name.upper(),
                    'Method': method.upper(),
                    'k': res['k'],
                    'ARI': res['external']['ARI'],
                    'NMI': res['external']['NMI'],
                    'AMI': res['external']['AMI'],
                    'FMI': res['external']['FMI'],
                    'V-Measure': res['external']['V-Measure']
                })
    
    external_df = pd.DataFrame(external_data) if external_data else None
    
    # Fairness Metrics Table
    fairness_data = []
    for dataset_name in dataset_names:
        for method in methods:
            if method in results[dataset_name]:
                res = results[dataset_name][method]
                fairness_data.append({
                    'Dataset': dataset_name.upper(),
                    'Method': method.upper(),
                    'k': res['k'],
                    'Balance': res['fairness']['Balance'],
                    'SPD': res['fairness']['SPD'],
                    'Disparate_Impact': res['fairness']['Disparate_Impact'],
                    'Entropy': res['fairness']['Entropy']
                })
    
    fairness_df = pd.DataFrame(fairness_data)
    
    return quality_df, external_df, fairness_df

## 4. OPTIMAL K SELECTION FOR EACH METHOD

In [12]:
def normalize_metric(values, inverse=False):
    values = np.array(values)
    min_val, max_val = np.min(values), np.max(values)
    
    if max_val - min_val == 0:
        return np.ones_like(values) * 0.5
    
    normalized = (values - min_val) / (max_val - min_val)
    return 1 - normalized if inverse else normalized


# Select optimize k
def find_optimal_k_for_method(X, sensitive_attr, method_name, k_range=(2, 11)):
    results = {
        'k_values': [],
        'sse': [],
        'silhouette': [],
        'calinski': [],
        'dbi': [],
        'silhouette_std': []
    }
    
    for k in range(k_range[0], k_range[1]):
        # Run clustering
        if method_name == 'fair_centroid':
            labels, centers = fair_centroid_clustering(X, sensitive_attr, k, max_iter=10)
        elif method_name == 'postprocessing_nfp':
            labels, centers = postprocessing_clustering_nfp(X, sensitive_attr, k, max_iter=15)
        elif method_name == 'postprocessing_gini':
            labels, centers = postprocessing_clustering_gini(X, sensitive_attr, k, max_iter=15)
        elif method_name == 'rawlsian':
            labels, centers = rawlsian_kmeans(X, sensitive_attr, k, n_runs=20)
        
        # Calculate metrics
        sse = np.sum((X - centers[labels]) ** 2)
        
        if len(np.unique(labels)) > 1:
            sil_score = silhouette_score(X, labels)
            sil_samples = silhouette_samples(X, labels)
            sil_std = np.std(sil_samples)
            ch_score = calinski_harabasz_score(X, labels)
            db_score = davies_bouldin_score(X, labels)
        else:
            sil_score = sil_std = ch_score = db_score = 0
        
        results['k_values'].append(k)
        results['sse'].append(sse)
        results['silhouette'].append(sil_score)
        results['calinski'].append(ch_score)
        results['dbi'].append(db_score)
        results['silhouette_std'].append(sil_std)
    
    # Normalize and combine
    norm_sse = normalize_metric(results['sse'], inverse=True)
    norm_sil = normalize_metric(results['silhouette'], inverse=False)
    norm_ch = normalize_metric(results['calinski'], inverse=False)
    norm_dbi = normalize_metric(results['dbi'], inverse=True)
    norm_sil_std = normalize_metric(results['silhouette_std'], inverse=True)
    
    combined_scores = (norm_sse + norm_sil + norm_ch + norm_dbi + norm_sil_std) / 5
    optimal_idx = np.argmax(combined_scores)
    optimal_k = results['k_values'][optimal_idx]
    
    results['combined_scores'] = combined_scores
    results['optimal_k'] = optimal_k
    
    return results


print("\n" + "="*80)
print("FINDING OPTIMAL K FOR ALL 6 METHODS")
print("="*80)

optimal_k_results = {}
methods = ['fair_centroid', 'postprocessing_nfp','postprocessing_gini', 'rawlsian']

for dataset_name in dataset_names:
    X = datasets[dataset_name]['X']
    sensitive = datasets[dataset_name]['sensitive']
    
    print(f"\n{dataset_name.upper()}:")
    optimal_k_results[dataset_name] = {}
    
    for method in methods:
        print(f"  Finding optimal k for {method.upper()}...", end='', flush=True)
        results = find_optimal_k_for_method(X, sensitive, method)
        optimal_k_results[dataset_name][method] = results
        print(f" k={results['optimal_k']}")


# Create optimal k summary table
print("\n" + "="*80)
print("OPTIMAL K VALUES FOR EACH METHOD")
print("="*80)

summary_data = []
for dataset_name in dataset_names:
    row = {'Dataset': dataset_name.upper()}
    for method in methods:
        row[f'{method.upper()}_k'] = optimal_k_results[dataset_name][method]['optimal_k']
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))
summary_df.to_csv('optimal_k_by_method.csv', index=False)
print("\n✓ Saved to: optimal_k_by_method.csv")


FINDING OPTIMAL K FOR EACH METHOD (APPROACH 1: COMBINED SCORE)

ADULT:
  Finding optimal k for KMEANS...
    Optimal k = 5
  Finding optimal k for FAIRLET...
    Optimal k = 3
  Finding optimal k for BFKM...
    Optimal k = 6

COMPAS:
  Finding optimal k for KMEANS...
    Optimal k = 2
  Finding optimal k for FAIRLET...
    Optimal k = 2
  Finding optimal k for BFKM...
    Optimal k = 2

GERMAN:
  Finding optimal k for KMEANS...
    Optimal k = 6
  Finding optimal k for FAIRLET...
    Optimal k = 3
  Finding optimal k for BFKM...
    Optimal k = 2

CREDIT:
  Finding optimal k for KMEANS...
    Optimal k = 2
  Finding optimal k for FAIRLET...
    Optimal k = 2
  Finding optimal k for BFKM...
    Optimal k = 2

LAW:
  Finding optimal k for KMEANS...
    Optimal k = 4
  Finding optimal k for FAIRLET...
    Optimal k = 2
  Finding optimal k for BFKM...
    Optimal k = 3

OPTIMAL K VALUES FOR EACH METHOD

Dataset  KMEANS_k  FAIRLET_k  BFKM_k
  ADULT         5          3       6
 COMPAS    

## 5. RUN CLUSTERING WITH OPTIMAL K FOR EACH METHOD

In [14]:
print("\n" + "="*80)
print("RUNNING CLUSTERING EXPERIMENTS WITH OPTIMAL K")
print("="*80)

final_results = {}

for dataset_name in dataset_names:
    X = datasets[dataset_name]['X']
    sensitive = datasets[dataset_name]['sensitive']
    
    print(f"\n{dataset_name.upper()}:")
    final_results[dataset_name] = {}
    
    for method in methods:
        optimal_k = optimal_k_results[dataset_name][method]['optimal_k']
        print(f"  {method.upper()} with k={optimal_k}...")
        
        # Run clustering
        if method == 'kmeans':
            labels, centers = standard_kmeans(X, optimal_k)
        elif method == 'fairlet':
            labels, centers = fairlet_clustering(X, sensitive, optimal_k)
        elif method == 'bfkm':
            labels, centers = bfkm_clustering(X, sensitive, optimal_k)
        
        # Calculate metrics
        quality = calculate_quality_metrics(X, labels)
        fairness = {
            'Balance': calculate_balance(labels, sensitive),
            'SPD': calculate_spd(labels, sensitive),
            'Disparate_Impact': calculate_disparate_impact(labels, sensitive),
            'Entropy': calculate_entropy(labels, sensitive)
        }
        
        final_results[dataset_name][method] = {
            'k': optimal_k,
            'labels': labels,
            'centers': centers,
            'quality': quality,
            'fairness': fairness
        }
        
        print(f"    Silhouette: {quality['Silhouette']:.4f}, Balance: {fairness['Balance']:.4f}")


RUNNING CLUSTERING EXPERIMENTS WITH OPTIMAL K

ADULT:
  KMEANS with k=5...
    Silhouette: 0.1584, Balance: 0.3723
  FAIRLET with k=3...
    Silhouette: 0.0271, Balance: 0.7667
  BFKM with k=6...
    Silhouette: 0.1157, Balance: 0.3501

COMPAS:
  KMEANS with k=2...
    Silhouette: 0.1794, Balance: 0.2623
  FAIRLET with k=2...
    Silhouette: 0.0127, Balance: 0.5762
  BFKM with k=2...
    Silhouette: 0.1793, Balance: 0.2622

GERMAN:
  KMEANS with k=6...
    Silhouette: 0.0913, Balance: 0.3840
  FAIRLET with k=3...
    Silhouette: -0.0010, Balance: 0.7378
  BFKM with k=2...
    Silhouette: 0.0811, Balance: 0.4559

CREDIT:
  KMEANS with k=2...
    Silhouette: 0.1383, Balance: 0.6422
  FAIRLET with k=2...
    Silhouette: 0.0380, Balance: 0.7444
  BFKM with k=2...
    Silhouette: 0.1383, Balance: 0.6422

LAW:
  KMEANS with k=4...
    Silhouette: 0.2817, Balance: 0.7211
  FAIRLET with k=2...
    Silhouette: 0.0967, Balance: 0.8214
  BFKM with k=3...
    Silhouette: 0.2625, Balance: 0.6922


## 6. SAVE RESULTS

In [16]:
with open('final_results_optimal_k_per_method.pkl', 'wb') as f:
    pickle.dump({
        'optimal_k_results': optimal_k_results,
        'final_results': final_results,
        'datasets': dataset_names
    }, f)

print("\n✓ Results saved to: final_results_optimal_k_per_method.pkl")


✓ Results saved to: final_results_optimal_k_per_method.pkl


## 7. CREATE COMPARISON TABLES

In [34]:
print("\n" + "="*80)
print("CREATING COMPARISON TABLES")
print("="*80)

# Quality metrics table
quality_data = []
for dataset_name in dataset_names:
    for method in methods:
        res = final_results[dataset_name][method]
        quality_data.append({
            'Dataset': dataset_name.upper(),
            'Method': method.upper(),
            'k': res['k'],
            'Inertia': res['quality']['Inertia'],
            'Silhouette': res['quality']['Silhouette'],
            'Calinski_Harabasz': res['quality']['Calinski_Harabasz'],
            'Davies_Bouldin': res['quality']['Davies_Bouldin'],
            'Dunn_Index': res['quality']['Dunn_Index'],
            'WCSS': res['quality']['WCSS'],
            'BCSS': res['quality']['BCSS'],
            'TSS': res['quality']['TSS'],
            'Variance_Ratio': res['quality']['Variance_Ratio'],
            'Inter_Cluster_Distance': res['quality']['Inter_Cluster_Distance'],
            'Intra_Cluster_Distance': res['quality']['Intra_Cluster_Distance'],
            'Separation_Index': res['quality']['Separation_Index']
        })

quality_df = pd.DataFrame(quality_data)
print("\n Quality Metrics:")
print(quality_df.to_string(index=False))
quality_df.to_csv('quality_metrics_optimal_k.csv', index=False)

# External metrics table
external_data = []
for dataset_name in dataset_names:
    for method in methods:
        res = final_results[dataset_name][method]
        ext = res.get('external', {})  # 如果不存在则返回空字典
        external_data.append({
            'Dataset': dataset_name.upper(),
            'Method': method.upper(),
            'k': res['k'],
            'ARI': ext.get('ARI', 0),
            'NMI': ext.get('NMI', 0),
            'AMI': ext.get('AMI', 0),
            'FMI': ext.get('FMI', 0),
            'V-Measure': ext.get('V-Measure', 0)
        })

external_df = pd.DataFrame(external_data)
print("\nExternal Metrics:")
print(external_df.to_string(index=False))
external_df.to_csv('external_metrics_optimal_k.csv', index=False)

# Fairness metrics table
fairness_data = []
for dataset_name in dataset_names:
    for method in methods:
        res = final_results[dataset_name][method]
        fairness_data.append({
            'Dataset': dataset_name.upper(),
            'Method': method.upper(),
            'k': res['k'],
            'Balance': res['fairness']['Balance'],
            'SPD': res['fairness']['SPD'],
            'Disparate Impact': res['fairness']['Disparate_Impact'],
            'Entropy': res['fairness']['Entropy']
        })

fairness_df = pd.DataFrame(fairness_data)
print("\nFairness Metrics:")
print(fairness_df.to_string(index=False))
fairness_df.to_csv('fairness_metrics_optimal_k.csv', index=False)

print("\n✓ Tables saved!")



CREATING COMPARISON TABLES

Quality Metrics:
Dataset  Method  k       Inertia  Silhouette  Calinski_Harabasz  Davies_Bouldin  Dunn_Index          WCSS          BCSS      TSS  Variance_Ratio  Inter_Cluster_Distance  Intra_Cluster_Distance  Separation_Index
  ADULT  KMEANS  5 243602.073317    0.158367        3662.568871        1.592339           0 243602.073317 118341.926683 361944.0        0.485800                       0                       0                 0
  ADULT FAIRLET  3 343669.793131    0.027102         801.833353        4.202997           0 343669.793131  18274.206869 361944.0        0.053174                       0                       0                 0
  ADULT    BFKM  6 255462.589785    0.115743        2513.912827        2.134232           0 255462.589785 106481.410215 361944.0        0.416818                       0                       0                 0
 COMPAS  KMEANS  2 193334.994110    0.179372        2018.182363        1.996162           0 193334.994110  415